In [1]:
import json
from collections import defaultdict

In [2]:
with open('./outputs/alphafold.json', 'r') as file:
    data = json.load(file)

In [3]:
fan_in = defaultdict(int)
fan_out = defaultdict(int)

for module_name, module_info in data.items():
    if "imported_by" in module_info:
        fan_in[module_name] = len(module_info["imported_by"])
    else:
        fan_in[module_name] = 0

    if "imports" in module_info:
        fan_out[module_name] = len(module_info["imports"])
    else:
        fan_out[module_name] = 0

print(f"{'Module':<30} {'Fan-In':<10} {'Fan-Out':<10}")
print("-" * 50)
for module_name in sorted(data.keys()):
    print(f"{module_name:<30} {fan_in[module_name]:<10} {fan_out[module_name]:<10}")

Module                         Fan-In     Fan-Out   
--------------------------------------------------
alphafold                      5          0         
alphafold.common               5          6         
alphafold.data                 2          7         
alphafold.model                3          4         
alphafold.relax                2          5         
alphafold.version              1          0         
numpy                          24         19        
numpy.__config__               1          2         
numpy._distributor_init        1          1         
numpy._globals                 2          0         
numpy._pytesttester            10         2         
numpy.compat                   5          2         
numpy.core                     11         8         
numpy.ctypeslib                1          2         
numpy.dtypes                   2          2         
numpy.exceptions               2          0         
numpy.fft                      2          4     

In [4]:
# Identify Highly Coupled Modules and Their Dependencies
print("\n=== Highly Coupled Modules and Their Dependencies ===")

# Define a threshold for "highly coupled" (e.g., fan-in + fan-out > 10)
threshold = 10
highly_coupled = []

for module_name in data.keys():
    total_coupling = fan_in[module_name] + fan_out[module_name]
    if total_coupling > threshold:
        highly_coupled.append((module_name, total_coupling, fan_in[module_name], fan_out[module_name]))

highly_coupled.sort(key=lambda x: x[1], reverse=True)

for module_name, total, fin, fout in highly_coupled:
    print(f"\nModule: {module_name}")
    print(f"Total Coupling (Fan-In + Fan-Out): {total} (Fan-In: {fin}, Fan-Out: {fout})")
    print("Imported by:", data[module_name].get("imported_by", []))
    print("Imports:", data[module_name].get("imports", []))


=== Highly Coupled Modules and Their Dependencies ===

Module: numpy
Total Coupling (Fan-In + Fan-Out): 43 (Fan-In: 24, Fan-Out: 19)
Imported by: ['alphafold.common', 'alphafold.data', 'alphafold.model', 'alphafold.relax', 'numpy', 'numpy.__config__', 'numpy._distributor_init', 'numpy._pytesttester', 'numpy.compat', 'numpy.core', 'numpy.ctypeslib', 'numpy.dtypes', 'numpy.fft', 'numpy.lib', 'numpy.linalg', 'numpy.ma', 'numpy.matrixlib', 'numpy.polynomial', 'numpy.random', 'numpy.testing', 'run_alphafold.py', 'scipy', 'scipy.linalg', 'scipy.special']
Imports: ['numpy', 'numpy.__config__', 'numpy._distributor_init', 'numpy._globals', 'numpy._pytesttester', 'numpy.compat', 'numpy.core', 'numpy.ctypeslib', 'numpy.dtypes', 'numpy.exceptions', 'numpy.fft', 'numpy.lib', 'numpy.linalg', 'numpy.ma', 'numpy.matrixlib', 'numpy.polynomial', 'numpy.random', 'numpy.testing', 'numpy.version']

Module: numpy.core
Total Coupling (Fan-In + Fan-Out): 19 (Fan-In: 11, Fan-Out: 8)
Imported by: ['numpy', 'nu

In [5]:
print("\n=== Cyclic Dependencies ===")
# Build a directed graph as an adjacency list
graph = defaultdict(list)
for module_name, module_info in data.items():
    if "imports" in module_info:
        for imported_module in module_info["imports"]:
            graph[module_name].append(imported_module)

# Function to detect cycles using DFS
def find_cycles(graph, all_modules):
    visited = set()
    rec_stack = set()
    cycles = []

    def dfs(node, path):
        visited.add(node)
        rec_stack.add(node)
        path.append(node)

        # Use .get() to safely access neighbors without modifying graph
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor, path.copy())
            elif neighbor in rec_stack:
                # Cycle detected
                cycle_start = path.index(neighbor)
                cycle = path[cycle_start:]
                cycle.append(neighbor)  # Close the cycle
                cycles.append(cycle)

        rec_stack.remove(node)

    # Iterate over all modules, not just graph keys
    for node in all_modules:
        if node not in visited:
            dfs(node, [])

    return cycles

# Find and print cycles
all_modules = list(data.keys())
cycles = find_cycles(graph, all_modules)
if cycles:
    print("Cyclic Dependencies Detected:")
    for cycle in cycles:
        print(" -> ".join(cycle))
else:
    print("No Cyclic Dependencies Detected.")


=== Cyclic Dependencies ===
Cyclic Dependencies Detected:
alphafold.common -> alphafold.common
numpy -> numpy
numpy -> numpy.__config__ -> numpy
numpy -> numpy.__config__ -> numpy.core -> numpy
numpy -> numpy.__config__ -> numpy.core -> numpy._pytesttester -> numpy
numpy -> numpy.__config__ -> numpy.core -> numpy._pytesttester -> numpy.testing -> numpy
numpy._pytesttester -> numpy.testing -> numpy._pytesttester
numpy.core -> numpy._pytesttester -> numpy.testing -> numpy.core
numpy -> numpy.__config__ -> numpy.core -> numpy._pytesttester -> numpy.testing -> numpy.lib -> numpy
numpy._pytesttester -> numpy.testing -> numpy.lib -> numpy._pytesttester
numpy -> numpy.__config__ -> numpy.core -> numpy._pytesttester -> numpy.testing -> numpy.lib -> numpy.compat -> numpy
numpy.compat -> numpy.compat
numpy.core -> numpy._pytesttester -> numpy.testing -> numpy.lib -> numpy.core
numpy.lib -> numpy.lib
numpy -> numpy.__config__ -> numpy.core -> numpy._pytesttester -> numpy.testing -> numpy.lib -> 

In [6]:
# Explanation of impact on maintainability
print("\nImpact on Maintainability:")
print("- Cyclic dependencies make the system harder to understand and modify because changes in one module can have unpredictable effects on others in the cycle.")
print("- They can lead to runtime errors (e.g., import errors if a module is accessed before it's fully initialized).")
print("- Refactoring becomes challenging because you can't easily isolate a module for modification without breaking the cycle.")


# Check for Unused and Disconnected Modules
print("\n=== Unused/Disconnected Modules ===")

# Disconnected modules: Fan-in = 0 and Fan-out = 0
disconnected_modules = [module for module in data.keys() if fan_in[module] == 0 and fan_out[module] == 0]
print("Disconnected Modules (Fan-In = 0 and Fan-Out = 0):", disconnected_modules if disconnected_modules else "None")


Impact on Maintainability:
- Cyclic dependencies make the system harder to understand and modify because changes in one module can have unpredictable effects on others in the cycle.
- They can lead to runtime errors (e.g., import errors if a module is accessed before it's fully initialized).
- Refactoring becomes challenging because you can't easily isolate a module for modification without breaking the cycle.

=== Unused/Disconnected Modules ===
Disconnected Modules (Fan-In = 0 and Fan-Out = 0): None


In [7]:
# Assess the Depth of Dependencies
print("\n=== Depth of Dependencies ===")
# Function to calculate the longest path (depth) from a starting node using DFS
def calculate_depth(graph, start):
    visited = set()
    max_depth = 0

    def dfs(node, depth):
        nonlocal max_depth
        visited.add(node)
        max_depth = max(max_depth, depth)

        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs(neighbor, depth + 1)

    dfs(start, 0)
    return max_depth

# Calculate depth starting from the entry point (run_alphafold.py)
entry_point = "run_alphafold.py"
if entry_point in graph:
    depth= calculate_depth(graph, entry_point)
    print(f"Maximum Dependency Depth from {entry_point}: {depth}")
else:
    print(f"Entry point {entry_point} not found in the graph.")

# Find the maximum depth across all nodes
max_depth_all = 0
for node in graph:
    depth= calculate_depth(graph, node)
    max_depth_all = max(max_depth_all, depth)
print(f"Maximum Dependency Depth in the Entire Graph: {max_depth_all}")


=== Depth of Dependencies ===
Maximum Dependency Depth from run_alphafold.py: 8
Maximum Dependency Depth in the Entire Graph: 8
